# 💾 Day 5 Lab: The Recomputation Trap & Explicit Cache Tuning

Welcome to Day 5. Today, we measure the performance tax of un-cached data scans during recursion. We will execute an iterative tree traversal over an un-cached base table, record the timings, and then apply explicit state controls to observe how pre-caching eliminates redundant processing cycles.

### Roadmap:
* **1. Hierarchical Data Assembly:** Generating a deep, programmatically expanding corporate hierarchy.
* **2. The Recomputation Trap:** Running recursion over a raw, un-cached table view to witness performance degradation.
* **3. Explicit Cache Tuning:** Persisting the base relationship in memory to eliminate redundant lookups.

### Initialize your Spark session

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('Recursive-CTE-Day5').config('spark.sql.ansi.enabled', 'true').getOrCreate()
print(f'Current Spark Version: {spark.version}')

Current Spark Version: 4.1.2


### CASE 1: Hierarchical Data Assembly
Let us build a structured corporate organization tree containing thousands of distinct nodes.

In [2]:
print('--- Case 1: Programmatically generating corporate data structure ---')
start_gen = time.time()

records = [('EMP_0', 'CEO', None)]
current_id = 1
nodes_per_level = 4
max_depth_layers = 7

parents = [0]
for depth in range(1, max_depth_layers + 1):
    next_parents = []
    for p in parents:
        for _ in range(nodes_per_level):
            records.append((f'EMP_{current_id}', f'Employee_{current_id}', f'EMP_{p}'))
            next_parents.append(current_id)
            current_id += 1
    parents = next_parents

raw_df = spark.createDataFrame(records, ['employee_id', 'employee_name', 'manager_id'])
raw_df.createOrReplaceTempView('raw_employees')
print(f'Generated data structure with {raw_df.count()} unique corporate records.')
print(f'Data Assembly Duration: {time.time() - start_gen:.4f} seconds')

--- Case 1: Programmatically generating corporate data structure ---
Generated data structure with 21845 unique corporate records.
Data Assembly Duration: 20.3662 seconds


### CASE 2: The Recomputation Trap (Un-cached Execution)
We run the recursive hierarchy query over our raw, un-cached data view. Watch your cluster metrics closely: every single level of depth forces Spark to re-evaluate the base table relation scan.

In [3]:
print('\n--- Case 2: Running recursion over an un-cached data layout ---')
start_uncached = time.time()

uncached_result_df = spark.sql('''
WITH RECURSIVE uncached_reports AS (
  SELECT employee_id, manager_id, 0 AS depth, array(employee_id) AS path
  FROM raw_employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.manager_id, ur.depth + 1 AS depth, concat(ur.path, array(e.employee_id)) AS path
  FROM raw_employees e
  JOIN uncached_reports ur ON e.manager_id = ur.employee_id
  WHERE ur.depth < 20 AND NOT array_contains(ur.path, e.employee_id)
)
SELECT depth, count(*) AS layer_count 
FROM uncached_reports 
GROUP BY depth 
ORDER BY depth
''')

uncached_result_df.show()
duration_uncached = time.time() - start_uncached
print(f'Un-cached Traversal Total Time: {duration_uncached:.4f} seconds')


--- Case 2: Running recursion over an un-cached data layout ---
+-----+-----------+
|depth|layer_count|
+-----+-----------+
|    0|          1|
|    1|          4|
|    2|         16|
|    3|         64|
|    4|        256|
|    5|       1024|
|    6|       4096|
|    7|      16384|
+-----+-----------+

Un-cached Traversal Total Time: 294.4022 seconds


### CASE 3: Explicit Cache Tuning (Optimized Run)
Now we apply our tuning playbook rule. We explicitly cache our base dataset in memory and trigger an evaluation action to pin it before running the exact same recursive query block.

In [4]:
print('\n--- Case 3: Optimizing execution with explicit pre-caching states ---')
start_cached = time.time()

# Force data persistence and pin the relation inside memory tables
raw_df.cache()
raw_df.count() 

cached_result_df = spark.sql('''
WITH RECURSIVE cached_reports AS (
  SELECT employee_id, manager_id, 0 AS depth, array(employee_id) AS path
  FROM raw_employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.manager_id, cr.depth + 1 AS depth, concat(cr.path, array(e.employee_id)) AS path
  FROM raw_employees e
  JOIN cached_reports cr ON e.manager_id = cr.employee_id
  WHERE cr.depth < 20 AND NOT array_contains(cr.path, e.employee_id)
)
SELECT depth, count(*) AS layer_count 
FROM cached_reports 
GROUP BY depth 
ORDER BY depth
''')

cached_result_df.show()
duration_cached = time.time() - start_cached
print(f'Optimized Cached Traversal Total Time: {duration_cached:.4f} seconds')
print(f'Total Performance Improvement Factor: {duration_uncached / duration_cached:.2f}x')


--- Case 3: Optimizing execution with explicit pre-caching states ---
+-----+-----------+
|depth|layer_count|
+-----+-----------+
|    0|          1|
|    1|          4|
|    2|         16|
|    3|         64|
|    4|        256|
|    5|       1024|
|    6|       4096|
|    7|      16384|
+-----+-----------+

Optimized Cached Traversal Total Time: 41.0844 seconds
Total Performance Improvement Factor: 7.17x


## 📊 Post-Lab Analysis: The Persisted State Dividend

By evaluating the execution durations of our un-cached baseline against our tuned optimized pipeline, we have captured a pristine, quantifiable look at how `UnionLoopExec` manages state memory.

### 🛑 Performance Telemetry Reflections
* **The Un-cached Trap (Case 2):** **294.4022 seconds**
  When the source table containing **21,845 records** was left completely raw, Spark had no method to preserve the structural base table relationship between iterations. For every single one of our 8 layers of tree depth, the system had to spawn independent tasks to repeatedly open file connections, deserialize data bytes, and re-compile the base relation scan. Framework overhead and redundant storage I/O caused our runtime to drag for nearly 5 full minutes.
* **The Persistent State (Case 3):** **41.0844 seconds**
  Explicitly invoking `.cache()` and forcing a `.count()` to pin the base dataset before starting the loop immediately broke the recomputation loop. While `UnionLoopExec` still built its dynamic iteration plans and tracked loop execution boundaries via internal shuffle files sequentially, the base table scan branch shifted from an expensive, recurring I/O scan into an instantaneous, in-memory reference look-up.

### 📈 Performance Dividend Summary
By taking programmatic control of the data lifecycle and pinning the base table state, we slashed execution time from **294.4022 seconds** down to **41.0844 seconds**, securing an immediate **7.17x performance improvement factor**.